# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditi-avni/ML-FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/aditi-avni/ML-FlyRank.git

Cloning into 'ML-FlyRank'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 135 (delta 47), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.85 MiB | 10.06 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:
import os

os.chdir("ML-FlyRank")
print(os.getcwd())

/content/ML-FlyRank


In [4]:
!pip install -q pandas numpy scikit-learn matplotlib duckdb

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 30000
Columns: 44


In [11]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [12]:


df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [13]:
df.head(3).T

,0,1,2
content_id,content_304f48230142,content_a1fb4e703a9e,content_9aa793d4d895
client_id,client_f369cb89fc,client_4e07408562,client_7f2253d7e2
search_volume,10.0,90.0,0.0
competition,0.67,0.01,0.0
competition_level,HIGH,LOW,LOW
cpc,2.05,0.05,0.0
content_type,keyword article,keyword article,keyword article
main_intent,transactional,informational,informational
word_count,3221.0,2481.0,3515.0
char_count,20457.0,15562.0,23643.0


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one pseudonymized content item for one client. The dataset contains overall 90-day search and traffic performance, along with separate previous-30-day and last-30-day performance windows. These windows allow us to observe recent changes in page performance.

In [20]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

Rows: 30000
Columns: 44
Unique content IDs: 30000
Unique clients: 32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Features: page/content characteristics and observed performance signals such as search volume, competition, content type, intent, word count, impressions, clicks, sessions, content age, freshness, CTR, average position, engagement rate and scroll rate.

Label: trend_direction, which represents the observed performance trend. trend_pct is also excluded because it is used to define the trend.

Context: content_id and client_id, which identify the content and client and can be used for grouping or splitting but should not be model features.

Excluded: trend_direction and trend_pct from the feature set because they directly define the outcome we want to learn. IDs are also excluded from model features because they are identifiers rather than meaningful page characteristics.

In [21]:
print("Feature examples:")
print([
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
])

print("\nLabel:")
print(["trend_direction"])

print("\nContext:")
print(["content_id", "client_id"])

print("\nExcluded:")
print(["trend_direction", "trend_pct"])

Feature examples:
['search_volume', 'competition', 'word_count', 'impressions_90d', 'clicks_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Label:
['trend_direction']

Context:
['content_id', 'client_id']

Excluded:
['trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())

print("\nTrend distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMissing values:")
missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset can show observed relationships between page characteristics, search performance and performance trends, but it cannot establish causal relationships. The 90-day and 30-day windows summarize historical performance rather than providing a controlled experiment. Missing values are not necessarily zero and may follow content-type patterns. Therefore, results will be treated as observed, directional and decision-support evidence rather than proof of what causes search performance changes or how Google's algorithm works.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.